In [ ]:
import numpy as np
import pandas as pd
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split
from torchvision.transforms.functional import to_tensor
import torch
from torch.utils.data import DataLoader, Dataset
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import warnings
import torch.nn.functional as F

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
from torch.utils.data import DataLoader

# Define transforms - COMPLETE THE MISSING PARTS
transform = transforms.Compose([
    # TODO: Resize to 28x28
    transforms.Resize((240,240)),
    transforms.Grayscale(3),  # Convert grayscale to RGB (Don't Touch!!)
    # TODO: Convert to Tensor
    transforms.ToTensor(),
    # TODO: Normalize with ImageNet mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225]
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load EMNIST letters dataset (given)
train_dataset = EMNIST(root='./data', split='letters', train=True, download=True, transform=transform)
test_dataset = EMNIST(root='./data', split='letters', train=False, download=True, transform=transform)

# Note: EMNIST letters has labels 1-26 (A-Z), so we have 26 classes
num_classes = 26

print(f"Training samples: {len(train_dataset)}")
print(f"Testing samples: {len(test_dataset)}")
print(f"Number of classes: {num_classes}")

In [ ]:
from IPython.testing import test
# Letter mapping (labels are 1-26 for A-Z)
letters = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

# Create DataLoaders and display samples
train_dataloader = DataLoader(train_dataset, 128, True)
test_dataloader = DataLoader(test_dataset, 128, False)

# Write your code here
import matplotlib.pyplot as plt
import numpy as np

# Get a batch of training data
data_iter = iter(train_dataloader)
images, labels = next(data_iter)


classes = letters

# Show images
fig, axes = plt.subplots(2, 5, figsize=(10, 5))
for i, ax in enumerate(axes.flat):
    img = images[i]
    img = np.transpose(img.numpy(), (1, 2, 0))  # Convert (C, H, W) to (H, W, C)

    ax.imshow(img)
    ax.set_title(classes[labels[i].item()])
    ax.axis("off")

plt.show()

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_v2_s

# Write your code here
import torch
import torch.nn as nn
from torchvision import models
from torchvision.models import efficientnet_v2_m
from torchvision.models import efficientnet_v2_s
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 2
model = models.efficientnet_v2_s()

criterion = nn.CrossEntropyLoss()

for param in model.parameters():
    param.requires_grad = False

for param in model.classifier.parameters():
    param.requires_grad = True

model = model.to(device)
optimizer = optim.AdamW(model.parameters(), weight_decay=0.01)

In [ ]:
# Write your code here
def train_one_epoch(model, optimizer, criterion,train_loader, device):
    model.train()

    total_loss = 0.0
    correct = 0
    total_samples = 0


    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss+= loss.item()

        predictions = outputs.argmax(dim=1)
        correct+= (predictions == labels).sum().item()
        total_samples += labels.shape[0]

    avg_loss = total_loss/len(train_loader)
    accuracy = correct/total_samples
    return avg_loss, accuracy

def validate_one_epoch(model, criterion, test_loader, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total_samples = 0
    with torch.no_grad():

        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            predictions = outputs.argmax(dim=1)
            correct+= (predictions == labels).sum().item()
            total_samples += labels.shape[0]

            total_loss+= loss.item()

    avg_loss = total_loss/len(test_loader)
    accuracy = correct/total_samples

    return avg_loss, accuracy

In [ ]:
num_epochs = 3
# 10 epochs is alot
print(device)
train_losses = []
train_accuracies = []
val_losses = []
val_accuracies = []
for epoch in range(num_epochs):
    avg_loss_train, train_accuracy = train_one_epoch(model, optimizer, criterion, train_dataloader, device)

    avg_loss_validation, val_accuracy = validate_one_epoch(model, criterion, test_dataloader, device)
    train_losses.append(avg_loss_train)
    train_accuracies.append(train_accuracy)
    val_losses.append(avg_loss_validation)
    val_accuracies.append(val_accuracy)
    print(f'Epoch [{epoch+1}/{num_epochs}], Train Loss: {avg_loss_train:.4f}, Train accuracy: {train_accuracy:.4f} ,Val Loss: {avg_loss_validation:.4f}, Val Accuracy: {val_accuracy:.4f}')


In [ ]:
import matplotlib.pyplot as plt

# Plot loss curve
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, num_epochs+1), train_losses, label="Train Loss", marker='o')
plt.plot(range(1, num_epochs+1), val_losses, label="Validation Loss", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Loss Curve")
plt.legend()

# Plot accuracy curve
plt.subplot(1, 2, 2)
plt.plot(range(1, num_epochs+1), train_accuracies, label="Train Accuracy", marker='o')
plt.plot(range(1, num_epochs+1), val_accuracies, label="Validation Accuracy", marker='o')
plt.xlabel("Epochs")
plt.ylabel("Accuracy (%)")
plt.title("Accuracy Curve")
plt.legend()

plt.show()

In [ ]:
# Write your code here
# predictions but not flipped and not from the function
def validate_one_epoch(model, criterion, test_loader, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total_samples = 0
    with torch.no_grad():  # Disable gradient calculation

        for images, labels in test_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            predictions = outputs.argmax(dim=1)
            h_flipped = torch.flip(images, dims=[3])
            v_flipped = torch.flip(images, dims=[2])
            correct+= (predictions == labels).sum().item()
            total_samples += labels.shape[0]

            total_loss+= loss.item()

    avg_loss = total_loss/len(test_loader)
    accuracy = correct/total_samples

    return avg_loss, accuracy